## GeoProxy discovery mode: search the whole Earth for new Moon/Mars terrain analogs

**Question answered:** *which places on Earth, including ones nobody has proposed, have terrain most like a chosen Moon or Mars site, and how sure are we?*

**Method: coarse-to-fine search, with built-in validation**

| Stage | What happens | Data |
|---|---|---|
| **A · Screen** | Earth's land is cut into ~350,000 windows of 20 km × 20 km. Each gets relief, slope and roughness at 1 km pixels | NOAA ETOPO 2022, 30 arc-seconds (its land heights come largely from Copernicus DEM) |
| **B · Confirm** | The best, well-separated candidates are re-measured at 500 m and re-ranked | Copernicus DEM GLO-90 (same as the main pipeline) |
| **Context** | Rock type compared with the target's published geology; distance to known analog sites (novelty); climate | Macrostrat geological maps; published papers; NASA POWER |
| **Validation** | (1) do two elevation products at two resolutions agree? (ETOPO land is largely derived from Copernicus, so this tests processing consistency) (2) do candidates survive changes in feature weights? (3) where do the established analog sites rank? (4) does Stage B confirm Stage A? | all of the above |

**Scores are percentiles, not arbitrary numbers:** "closer than 99.9% of Earth's land windows".

**Run in two passes:**
1. `QUICK_SCAN = True` (default): only latitudes 30–40°N, streamed, about 5 minutes. Check that every PASS line passes.
2. Set `QUICK_SCAN = False` in cell 2, then Run all: the whole Earth (downloads 3.2 GB inside Colab, about 20–30 minutes).

**If something fails:** the cell number tells you where. Cell 5 = maths, cell 7 = data access, cells 8–11 = computation (each prints its own checks).

### 1 · Install
*Why:* `rasterio` reads GeoTIFF rasters; `reverse_geocoder` names the nearest town to a candidate (offline GeoNames data); `pycountry` turns country codes into names; `global-land-mask` is an offline ~1 km land/ocean mask, used so that land **below sea level** (Danakil Depression, Dead Sea shore, Death Valley) is recognised as land instead of being thrown away as "water".

In [ ]:
!pip -q install rasterio reverse_geocoder pycountry global-land-mask

### 2 · Settings: the only cell you normally edit
- **QUICK_SCAN**: `True` = a fast test on one latitude band; `False` = the full Earth.
- Window and grid: Stage A uses 1 km pixels (the global data's resolution), Stage B uses 500 m (the main pipeline's grid). Within each stage, Earth and the target are always measured the same way.
- `WEIGHTS`: importance of each terrain feature (normalised automatically).

In [ ]:
QUICK_SCAN = True
QUICK_LAT_RANGE = (30.0, 40.0)    # band used when QUICK_SCAN = True
LAT_RANGE = (-84.0, 84.0)         # full scan

WINDOW_KM = 20                    # side of every analysis window
PIXEL_A_M = 1000                  # Stage A screen grid
PIXEL_B_M = 500                   # Stage B confirmation grid (same as the main pipeline)
SPACING_KM = 20                   # distance between window centres (windows do not overlap)
BAND_DEG = 5                      # latitude band loaded into memory at a time

MIN_COVERAGE = 0.95               # minimum share of valid elevation cells
MAX_WATER_FRACTION = 0.05         # reject windows with more than 5% non-land cells (GLOBE land mask, NOT "elevation <= 0")
ICE_MIN_THICKNESS_M = 10          # surface minus bedrock above this = ice
EXCLUDE_ICE_SHEETS = True         # rank rock terrain, not ice-sheet surfaces
MAX_ICE_FRACTION = 0.5
MAX_GLACIER_FRACTION = 0.3        # drop candidates whose window is >30% snow/ice in ESA WorldCover (mountain glaciers)
MAX_WATER_COVER = 0.2             # drop candidates whose window is >20% water in ESA WorldCover (lakes the land mask counts as land)
MIN_BARE_FRACTION = 0.3           # pre-filter: candidates must be at least 30% bare ground (Moon/Mars are 100% bare); 0 disables
MAX_PREFILTER_TRIES = 250         # at most this many land-cover lookups per target while hunting bare candidates

WEIGHTS = {"relief_m": 1.0, "slope_deg": 1.0, "roughness_m": 1.0}
N_CANDIDATES = 20                 # candidates confirmed at 500 m per target (final order = 500 m result)
MIN_SEPARATION_KM = 150           # candidates must be this far apart (no near-duplicates)
TOP_FRACTION = 0.001              # "top 0.1% of Earth's land" used for the stability test
RECALL_FRACTION = 0.01            # "top 1%" used for the dataset-agreement recall test
STABILITY_DRAWS = 200             # random weight sets for the stability test
STABILITY_ALPHA = 4.0             # how strongly weights vary (Dirichlet concentration; lower = wilder)
FACTOR_HOLD, FACTOR_PARTIAL = 1.5, 2.0   # feature verdicts: Earth and target within x1.5 = holds, within x2 = partial
BARE_HOLD, BARE_PARTIAL = 0.7, 0.3       # bare-ground share (Moon/Mars are 100% bare): >=70% holds, >=30% partial
N_AGREEMENT = 40                  # windows used to cross-check ETOPO against Copernicus
SEED = 42

SAVE_TO_DRIVE = True              # full scan: also save to your Google Drive (asks for permission)
AUTO_DOWNLOAD = True              # full scan: download discoveries.json to your computer
DRIVE_OUT = "/content/drive/MyDrive/geoproxy/geoproxy-data/processed"

TARGETS = [
    {"id": "jezero-crater", "name": "Jezero Crater", "body": "mars", "lat": 18.38, "lon": 77.58},
    {"id": "gale-crater", "name": "Gale Crater", "body": "mars", "lat": -5.4, "lon": 137.8},
    {"id": "elysium-insight", "name": "Elysium Planitia", "body": "mars", "lat": 4.502, "lon": 135.623},
    {"id": "shackleton-crater", "name": "Shackleton Crater", "body": "moon", "lat": -89.667, "lon": 129.783},
    {"id": "apollo-11-tranquillitatis", "name": "Mare Tranquillitatis", "body": "moon", "lat": 0.674, "lon": 23.473},
]

# Rock types of the targets, from published papers. primary = main rock in the window; secondary = also present.
# Classes: volcanic, plutonic, metamorphic, sedimentary, unconsolidated. Edit only with a citation.
TARGET_GEOLOGY = {
    "jezero-crater": {"primary": ["volcanic", "plutonic"], "secondary": ["sedimentary"],
        "summary": "Igneous crater floor (olivine-rich cumulate 'Séítah', basaltic 'Máaz') altered by water; river-delta and lake sediments on the western side.",
        "citation": "Farley, K. A. et al. (2022). Aqueously altered igneous rocks sampled on the floor of Jezero crater, Mars. Science 377, eabo2196. doi:10.1126/science.abo2196"},
    "gale-crater": {"primary": ["sedimentary"], "secondary": ["unconsolidated"],
        "summary": "Lake and river sediments filling the crater and forming Aeolis Mons (Mount Sharp), later exhumed by wind.",
        "citation": "Grotzinger, J. P. et al. (2015). Deposition, exhumation, and paleoclimate of an ancient lake deposit, Gale crater, Mars. Science 350, aac7575. doi:10.1126/science.aac7575"},
    "elysium-insight": {"primary": ["volcanic"], "secondary": ["unconsolidated", "sedimentary"],
        "summary": "Early Amazonian-Hesperian basaltic lava flows under impact-fragmented regolith; older sedimentary rocks beneath.",
        "citation": "Golombek, M. et al. (2020). Geology of the InSight landing site on Mars. Nature Communications 11, 1014. doi:10.1038/s41467-020-14679-1"},
    "shackleton-crater": {"primary": ["plutonic"], "secondary": ["unconsolidated"],
        "summary": "Lunar highland crust; crater walls expose nearly pure anorthosite (a plagioclase-rich plutonic rock), plus impact ejecta.",
        "citation": "Gawronska, A. J. et al. (2020), Advances in Space Research (geologic context of Shackleton crater); Haruyama, J. et al. (2013), Geophysical Research Letters"},
    "apollo-11-tranquillitatis": {"primary": ["volcanic"], "secondary": ["unconsolidated"],
        "summary": "Mare basalt: volcanic rock of pyroxene, plagioclase, ilmenite and olivine, under fine regolith.",
        "citation": "Lunar Sample Preliminary Examination Team (1969). Preliminary examination of lunar samples from Apollo 11. Science 165, 1211."},
}

# Established analog sites (from the main pipeline), used for validation and to flag "already known" candidates
KNOWN_SITES = [
    ("mount-cameroon", "Mount Cameroon", 4.217, 9.167), ("atacama-yungay", "Yungay, Atacama", -24.08, -70.0),
    ("dry-valleys-vanda", "Wright Valley", -77.53, 161.58), ("haughton-crater", "Haughton crater", 75.367, -89.683),
    ("mauna-kea", "Mauna Kea", 19.821, -155.468), ("meteor-crater", "Meteor Crater", 35.027, -111.022),
    ("ries-crater", "Nördlinger Ries", 48.88, 10.56), ("askja-iceland", "Askja", 65.05, -16.75),
    ("lonar-crater", "Lonar crater", 19.976, 76.508), ("mdrs-utah", "Mars Desert Research Station", 38.406, -110.792),
    ("rio-tinto", "Río Tinto", 37.69, -6.59), ("pilbara-dresser", "Pilbara (Dresser Fm.)", -21.15, 119.42),
    ("namib-sossusvlei", "Sossusvlei", -24.73, 15.29), ("vinson-massif", "Vinson Massif", -78.525, -85.617),
]

### 3 · Data sources
*Why:* one entry per dataset, also copied into the output JSON so every number can be traced. Everything is read **remotely, piece by piece**, except the full scan, which downloads ETOPO once inside Colab (not on your computer).

In [ ]:
BODIES = {"earth": {"radius_m": 6371000.0}, "mars": {"radius_m": 3396190.0}, "moon": {"radius_m": 1737400.0}}

ETOPO_BASE = "https://www.ngdc.noaa.gov/mgg/global/relief/ETOPO2022/data/30s"
ETOPO_URLS = {
    "surface": f"{ETOPO_BASE}/30s_surface_elev_gtif/ETOPO_2022_v1_30s_N90W180_surface.tif",
    "bed": f"{ETOPO_BASE}/30s_bed_elev_gtif/ETOPO_2022_v1_30s_N90W180_bed.tif",
}

SOURCES = {
    "etopo-2022-30s": {
        "body": "earth", "kind": "global_file", "native_res_m": 926, "path": None,
        "dataset": "ETOPO 2022 v1, 30 arc-second surface elevation (about 1 km)",
        "provider": "NOAA National Centers for Environmental Information",
        "url": ETOPO_URLS["surface"], "attribution": "NOAA NCEI (2022), ETOPO 2022 15 Arc-Second Global Relief Model, doi:10.25921/fd45-gt74",
        "caveat": "Land heights largely from Copernicus DEM; bathymetry in oceans (negative values).",
    },
    "etopo-2022-30s-bed": {
        "body": "earth", "kind": "global_file", "native_res_m": 926, "path": None,
        "dataset": "ETOPO 2022 v1, 30 arc-second bedrock elevation",
        "provider": "NOAA National Centers for Environmental Information",
        "url": ETOPO_URLS["bed"], "attribution": "NOAA NCEI (2022), ETOPO 2022, doi:10.25921/fd45-gt74",
        "caveat": "Differs from the surface only under the Antarctic and Greenland ice sheets.",
    },
    "copernicus-glo90": {
        "body": "earth", "kind": "earth_tiles", "native_res_m": 90,
        "dataset": "Copernicus DEM GLO-90 (about 90 m)",
        "provider": "ESA / European Union Copernicus programme, via AWS Open Data",
        "url": "https://registry.opendata.aws/copernicus-dem/",
        "attribution": "© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA",
        "caveat": "Surface model: includes vegetation canopy and buildings.",
    },
    "mola-463m": {
        "body": "mars", "kind": "global_file", "native_res_m": 463,
        "path": "/vsicurl/https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif",
        "dataset": "Mars MGS MOLA global DEM, 463 m", "provider": "NASA MGS MOLA (PDS); mosaic by USGS Astrogeology",
        "url": "https://planetarymaps.usgs.gov/mosaic/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif",
        "attribution": "NASA / MOLA Science Team / USGS Astrogeology", "caveat": "Interpolated where no laser tracks.",
    },
    "lola-118m": {
        "body": "moon", "kind": "global_file", "native_res_m": 118,
        "path": "/vsicurl/https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif",
        "dataset": "Moon LRO LOLA global DEM, 118 m", "provider": "NASA LRO LOLA (PDS); mosaic by USGS Astrogeology",
        "url": "https://planetarymaps.usgs.gov/mosaic/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif",
        "attribution": "NASA / LOLA Science Team / USGS Astrogeology", "caveat": "Heights relative to a 1737.4 km sphere.",
    },
    "nasa-power": {
        "body": "earth", "kind": "climate", "native_res_m": 50000,
        "dataset": "NASA POWER climatology (MERRA-2 based)", "provider": "NASA Langley Research Center POWER project",
        "url": "https://power.larc.nasa.gov/", "attribution": "NASA POWER Project",
        "caveat": "About 50 km grid: regional climate, not the exact window.",
    },
    "macrostrat": {
        "body": "earth", "kind": "geology", "native_res_m": None,
        "dataset": "Macrostrat geologic map units (compilation of published geological maps)",
        "provider": "Macrostrat, University of Wisconsin-Madison (CC-BY 4.0)",
        "url": "https://macrostrat.org/api/v2/geologic_units/map",
        "attribution": "Macrostrat (macrostrat.org); each unit also cites its original map",
        "caveat": "Point lookup at the window centre (not rock proportions); map detail varies by region.",
    },
    "esa-worldcover": {
        "body": "earth", "kind": "land_cover", "native_res_m": 10,
        "dataset": "ESA WorldCover 2021 v200 (10 m global land cover, 11 classes)",
        "provider": "European Space Agency, via AWS Open Data (CC-BY 4.0)",
        "url": "https://esa-worldcover.org/",
        "attribution": "© ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium",
        "caveat": "Read at ~100 m over the window's lat/lon box; snow/ice and water classes used to exclude glaciers and lakes; bare-ground share used as a pre-filter.",
    },
    "globe-land-mask": {
        "body": "earth", "kind": "land_mask", "native_res_m": 1000,
        "dataset": "Global ~1 km land/ocean mask (global-land-mask package, derived from NOAA GLOBE topography)",
        "provider": "NOAA GLOBE via the open-source global-land-mask package (MIT licence)",
        "url": "https://github.com/toddkarin/global-land-mask",
        "attribution": "Karin, Todd (2019). Global Land Mask. doi:10.5281/zenodo.4066722",
        "caveat": "Decides land vs ocean offline, so land below sea level is kept; some lakes count as land and are caught by the ESA WorldCover water share instead.",
    },
}
TERRAIN_SOURCE = {"mars": "mola-463m", "moon": "lola-118m"}
FEATURE_KEYS = ["relief_m", "slope_deg", "roughness_m"]

# eps ~= measurement noise floors (Copernicus GLO-90 vertical accuracy is a few metres; MOLA/LOLA cells are
# partly interpolated between laser tracks). log10(value + eps) saturates differences smaller than the floor,
# so "roughness 0.5 m vs 2 m" - both below what the data can actually resolve - no longer counts as signal.
# This matters most for very flat targets (Elysium Planitia: roughness ~1.9 m, slope ~0.26 deg).
LOG_EPS = {"relief_m": 5.0, "slope_deg": 0.2, "roughness_m": 2.0}

### 4 · Core functions
- Geometry (`aeqd_inverse`) and the terrain features are **identical to the main pipeline**, so results are comparable.
- `terrain_features_batch` measures thousands of windows at once (vectorised NumPy). Cell 5 proves it gives the same answer as the pipeline's one-window version.
- **Speed trick:** the shape of a window's lat/lon grid depends only on its latitude. So the geometry is computed once per row of windows, then shifted in longitude for every window in that row.
- **Similarity:** features are compared on a log scale (a 100 m vs 200 m difference matters as much as 1,000 m vs 2,000 m), each divided by **Earth's natural variability** (robust standard deviation over all land windows). Then a weighted distance is taken. **Score = percentile:** the share of Earth's land windows that are *less* similar.
- `elevation_skew` + `landform_class`: the skewness (lopsidedness) of a window's elevations separates **basin-like** terrain (crater, caldera, depression) from **peak-like** terrain (volcano, massif). Relief, slope and roughness are magnitude-only and cannot tell a crater from a volcano of the same size; this can.

In [ ]:
import math, os, time, json, warnings, subprocess, traceback
import numpy as np
from scipy.ndimage import map_coordinates
from global_land_mask import globe          # offline ~1 km land/ocean mask (no internet needed after install)
EARTH_KM_PER_DEG = math.pi * 6371.0 / 180

def aeqd_inverse(x, y, lat0, lon0, radius):
    """Metres (x east, y north) on an azimuthal equidistant map centred at (lat0, lon0) -> (lat, lon) degrees."""
    phi0, lam0 = math.radians(lat0), math.radians(lon0)
    x = np.asarray(x, dtype="float64"); y = np.asarray(y, dtype="float64")
    rho = np.hypot(x, y)
    c = rho / radius
    safe_rho = np.where(rho > 0, rho, 1.0)
    lat = np.arcsin(np.clip(np.cos(c) * math.sin(phi0) + np.where(rho > 0, y * np.sin(c) * math.cos(phi0) / safe_rho, 0.0), -1, 1))
    lon = lam0 + np.arctan2(x * np.sin(c), rho * math.cos(phi0) * np.cos(c) - y * math.sin(phi0) * np.sin(c))
    return np.degrees(lat), (np.degrees(lon) + 180.0) % 360.0 - 180.0

def is_land(lats, lons):
    """Offline land/ocean decision. Land below sea level counts as land (Danakil, Dead Sea shore, Death Valley)."""
    return globe.is_land(np.clip(np.asarray(lats, dtype="float64"), -89.99, 89.99),
                         np.clip(np.asarray(lons, dtype="float64"), -179.99, 179.99))

def haversine_km(lat1, lon1, lat2, lon2, radius_km=6371.0):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlam = np.radians(np.asarray(lon2, dtype="float64") - np.asarray(lon1, dtype="float64"))
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def window_n(pixel_m):
    return int(round(WINDOW_KM * 1000 / pixel_m)) + 1          # odd: the site is the centre pixel

def window_offsets(pixel_m, k):
    """x/y in metres of every sub-sample (k x k per pixel). Row 0 = north."""
    n = window_n(pixel_m)
    half = n * pixel_m / 2
    o = -half + (np.arange(n * k) + 0.5) * (pixel_m / k)
    X, Y = np.meshgrid(o, o[::-1])
    return X, Y

def supersample_factor(native_res_m, pixel_m):
    return max(1, min(6, math.ceil(pixel_m / native_res_m)))

def block_average(values, k):
    n = values.shape[-1] // k
    return values.reshape(values.shape[:-2] + (n, k, n, k)).mean(axis=(-3, -1))

def bilinear(data, rows, cols):
    return map_coordinates(data, [rows, cols], order=1, mode="nearest")

def _pct(a, q):
    return np.nanpercentile(a, q, axis=1) if np.isnan(a).any() else np.percentile(a, q, axis=1)

def _med(a):
    return np.nanmedian(a, axis=1) if np.isnan(a).any() else np.median(a, axis=1)

def terrain_features_batch(Z, pixel_m):
    """Z: (m, n, n) elevation windows -> (dict of (m,) feature arrays, (m,) coverage)."""
    Z = np.asarray(Z, dtype="float64")
    m = Z.shape[0]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        coverage = np.isfinite(Z).reshape(m, -1).mean(axis=1)
        gy = np.gradient(Z, pixel_m, axis=1)
        gx = np.gradient(Z, pixel_m, axis=2)
        slope = np.degrees(np.arctan(np.hypot(gx, gy))).reshape(m, -1)
        flat = Z.reshape(m, -1)
        relief = _pct(flat, 90) - _pct(flat, 10)
        c = Z[:, 1:-1, 1:-1]
        R_, C_ = Z.shape[1], Z.shape[2]
        tri = np.mean([np.abs(Z[:, 1 + dy:R_ - 1 + dy, 1 + dx:C_ - 1 + dx] - c)
                       for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)], axis=0)
        rough = np.nanmean(tri.reshape(m, -1), axis=1)
        slope_med = _med(slope)
    return {"relief_m": relief, "slope_deg": slope_med, "roughness_m": rough}, coverage

def terrain_features(z, pixel_m):
    f, cov = terrain_features_batch(np.asarray(z)[None], pixel_m)
    cov = float(cov[0])
    if cov < MIN_COVERAGE:
        return None, cov
    return {k: float(v[0]) for k, v in f.items()}, cov

def log_features(feats):
    """dict of scalars/arrays -> array (..., 3) of log10(value + eps)."""
    return np.stack([np.log10(np.asarray(feats[k], dtype="float64") + LOG_EPS[k]) for k in FEATURE_KEYS], axis=-1)

def robust_scale(L):
    med = np.median(L, axis=0)
    mad = 1.4826 * np.median(np.abs(L - med), axis=0)
    return np.where(mad > 0, mad, 1.0)

def weight_vector(weights=None):
    w = np.array([(weights or WEIGHTS)[k] for k in FEATURE_KEYS], dtype="float64")
    return w / w.sum()

def weighted_distance(L, t, scale, w):
    z = (np.asarray(L) - np.asarray(t)) / scale
    return np.sqrt(np.sum(w * z ** 2, axis=-1))

def percentile_closer(d_sorted, d):
    """% of windows that are LESS similar (larger distance) than d."""
    n = d_sorted.size
    return 100.0 * (n - np.searchsorted(d_sorted, d, side="right")) / n

def nms_select(order, lats, lons, min_sep_km, n_max):
    """Walk candidates best-first, keep one only if it is far from all kept ones (non-maximum suppression)."""
    chosen = []
    for i in order:
        if all(haversine_km(lats[i], lons[i], lats[j], lons[j]) >= min_sep_km for j in chosen):
            chosen.append(int(i))
            if len(chosen) == n_max:
                break
    return chosen

def feature_ratio(earth_value, target_value, key):
    """How many times larger/smaller Earth's value is than the target's (always >= 1), and the direction."""
    r = (earth_value + LOG_EPS[key]) / (target_value + LOG_EPS[key])
    return (r if r >= 1 else 1 / r), ("higher" if r >= 1 else "lower")

def verdict_from_factor(factor):
    return "holds" if factor <= FACTOR_HOLD else ("partial" if factor <= FACTOR_PARTIAL else "breaks")

def bare_verdict(fraction):
    return "holds" if fraction >= BARE_HOLD else ("partial" if fraction >= BARE_PARTIAL else "breaks")

def elevation_skew(z):
    """Skewness of the window's elevations: a few LOW cells pull it negative (basin/crater),
    a few HIGH cells pull it positive (peak/volcano). Zero-ish = flat or evenly tilted terrain."""
    v = np.asarray(z, dtype="float64")
    v = v[np.isfinite(v)]
    if v.size < 9:
        return 0.0
    s = v.std()
    if s == 0:
        return 0.0
    return float(np.mean(((v - v.mean()) / s) ** 3))

SKEW_BASIN, SKEW_PEAK = -0.3, 0.3

def landform_class(skew):
    return "basin-like" if skew <= SKEW_BASIN else ("peak-like" if skew >= SKEW_PEAK else "mixed/flat")

def landform_verdict(earth_skew, target_skew):
    """holds = same landform class as the target; partial = one of the two is mixed/flat; breaks = basin vs peak."""
    a, b = landform_class(earth_skew), landform_class(target_skew)
    if a == b:
        return "holds"
    if "mixed/flat" in (a, b):
        return "partial"
    return "breaks"

import re
ROCK_CLASSES = {   # keyword rules applied to a geological map unit's name / rock-type field
    "volcanic": r"basalt|lava|volcan|extrusive|scoria|tuff|andesit|rhyolit|dacit|pyroclast|cinder|ignimbrit|trachyt|hyaloclast",
    "plutonic": r"granit|granodiorit|diorit|gabbro|intrus|plutonic|batholit|batolit|anorthosit|syenit|tonalit|monzonit|peridotit",
    "metamorphic": r"gneiss|schist|metam|quartzite|marble|slate|phyllite|amphibolit|migmatit|metasediment|greenstone",
    "sedimentary": r"sandstone|shale|limestone|dolomit|dolostone|mudstone|siltstone|conglomerat|sedimentary|evaporit|gypsum|chert|marl|claystone|carbonate|lacustrine|clastic",
    "unconsolidated": r"alluvi|colluvi|\bdunes?\b|eolian|aeolian|\bsand\b|gravel|unconsolidated|surficial|\btill\b|moraine|loess|playa",
    "ice": r"glacier|\bice\b|icefield|permanent snow",
}

def rock_classes(text):
    t = (text or "").lower()
    return [c for c, pattern in ROCK_CLASSES.items() if re.search(pattern, t)]

def classify_unit(u):
    """Classify a map unit from its own name + rock-type field first; fall back to the stratigraphic name,
    then to the long description (which often mentions several rocks, so that case is labelled low confidence)."""
    for source, fields in (("name+lith", ("name", "lith")), ("strat_name", ("strat_name",)), ("description", ("descrip",))):
        classes = rock_classes(" ".join(str(u.get(k) or "") for k in fields))
        if classes:
            return classes, source
    return [], None

def geology_verdict(classes, target_geo):
    """holds = shares the target's main rock class; partial = shares a secondary class; breaks = neither."""
    if not classes or not target_geo:
        return "unknown"
    if set(classes) & set(target_geo["primary"]):
        return "holds"
    if set(classes) & set(target_geo.get("secondary", [])):
        return "partial"
    return "breaks"

### 5 · Self-test (no internet)
*Why:* proves the maths on cases with known answers, and proves the fast batch code equals the pipeline's original one-window code. If this fails, stop: the bug is in cell 4.

In [ ]:
from rasterio.transform import Affine

def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

# 1. geometry
R = BODIES["mars"]["radius_m"]; one_deg = R * math.radians(1)
la, lo = aeqd_inverse([0.0], [one_deg], 10, 20, R)
check("1 degree of arc north -> latitude +1", abs(la[0] - 11) < 1e-6 and abs(lo[0] - 20) < 1e-6)
la, lo = aeqd_inverse([one_deg], [0.0], 0, 20, R)
check("1 degree of arc east at the equator -> longitude +1", abs(la[0]) < 1e-6 and abs(lo[0] - 21) < 1e-6)
la, lo = aeqd_inverse([0.0], [one_deg], -90, 0, R)
check("works at the south pole", abs(la[0] + 89) < 1e-6)
Xs, Ys = window_offsets(PIXEL_A_M, 1)
la0, lo0 = aeqd_inverse(Xs, Ys, 37.0, 0.0, 6371000.0)
la1, lo1 = aeqd_inverse(Xs, Ys, 37.0, 55.0, 6371000.0)
check("speed trick valid: a window's grid only shifts in longitude",
      np.allclose(la0, la1) and np.allclose((lo0 + 55 + 180) % 360 - 180, lo1))
check("haversine: 1 degree of latitude = 111.19 km", abs(haversine_km(0, 0, 1, 0) - 111.19) < 0.01)

# 2. batch features == the pipeline's original one-window code
def reference_terrain_features(z, pixel_m):          # copied from geoproxy_pipeline.ipynb
    gy, gx = np.gradient(z, pixel_m)
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))
    c = z[1:-1, 1:-1]; rows, cols = z.shape
    diffs = [np.abs(z[1 + dy:rows - 1 + dy, 1 + dx:cols - 1 + dx] - c)
             for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)]
    tri = np.mean(diffs, axis=0)
    return {"relief_m": float(np.nanpercentile(z, 90) - np.nanpercentile(z, 10)),
            "slope_deg": float(np.nanmedian(slope)), "roughness_m": float(np.nanmean(tri))}

rng = np.random.default_rng(0)
n = window_n(PIXEL_A_M)
batch = np.cumsum(rng.normal(0, 30, (6, n, n)), axis=2)     # random rough terrain
batch[2, 3, 4] = np.nan                                      # one window with a hole
fb, _ = terrain_features_batch(batch, PIXEL_A_M)
same = all(abs(fb[k][i] - reference_terrain_features(batch[i], PIXEL_A_M)[k]) < 1e-9
           for i in range(6) for k in FEATURE_KEYS)
check("batch features identical to the pipeline's code (6 random windows, one with a hole)", same)
x = (np.arange(n) - n // 2) * PIXEL_A_M
Xp, _ = np.meshgrid(x, x)
fp, _ = terrain_features(0.1 * Xp, PIXEL_A_M)
check("10% plane: slope 5.71 degrees", abs(fp["slope_deg"] - math.degrees(math.atan(0.1))) < 1e-9)
check("10% plane: roughness = 0.75 x height step", abs(fp["roughness_m"] - 0.75 * 0.1 * PIXEL_A_M) < 1e-9)
check("block average (batched)", np.allclose(block_average(np.arange(32.0).reshape(2, 4, 4), 2)[1, 0, 0], 18.5))

# 3. global-grid sampling: a tilted plane must be interpolated exactly, and longitude wraps
data = np.fromfunction(lambda r, c: (-179.5 + c) * 2 + (89.5 - r), (180, 360))
tr = Affine(1.0, 0, -180.0, 0, -1.0, 90.0)
pad = 4
g_data = np.concatenate([data[:, -pad:], data, data[:, :pad]], axis=1)
rows = (np.array([10.25]) - tr.f) / tr.e - 0.5
cols = (np.array([20.75]) - tr.c) / tr.a - 0.5 + pad
check("bilinear sampling of a plane is exact", abs(bilinear(g_data, rows, cols)[0] - (20.75 * 2 + 10.25)) < 1e-9)
cols_w = (np.array([179.9]) - tr.c) / tr.a - 0.5 + pad
vw = bilinear(g_data, rows, cols_w)[0]
check("longitude wrap uses the other side of the map", np.isfinite(vw) and vw < data[:, -1].max())

# 4. similarity
L = np.array([[1.0, 2.0, 3.0], [1.5, 2.0, 3.0], [3.0, 0.0, 1.0]])
s = np.ones(3); w = weight_vector()
check("identical features -> distance 0", weighted_distance(L[0], L[0], s, w) == 0)
d_sorted = np.sort(np.array([1.0, 2.0, 3.0, 4.0]))
check("percentile: d=2.5 is closer than 50% of windows", percentile_closer(d_sorted, 2.5) == 50.0)
check("zero relief/slope/roughness gives finite logs", np.all(np.isfinite(log_features({"relief_m": 0, "slope_deg": 0, "roughness_m": 0}))))
la_ = np.array([0.0, 0.5, 5.0]); lo_ = np.array([0.0, 0.0, 0.0])
check("non-maximum suppression drops a candidate 55 km from a better one", nms_select([0, 1, 2], la_, lo_, 150, 3) == [0, 2])

# 5. geology classification (real Macrostrat unit names)
check("'Scoria cones' -> volcanic", rock_classes("Scoria cones") == ["volcanic"])
check("'Cretaceous Granitic' -> plutonic", rock_classes("Cretaceous Granitic") == ["plutonic"])
check("'Mesozoic-Cenozoic intrusive rocks' -> plutonic", rock_classes("Mesozoic-Cenozoic intrusive rocks") == ["plutonic"])
check("sandstone is sedimentary, not loose sand", rock_classes("Jurassic sandstone") == ["sedimentary"])
check("'Quaternary alluvium' -> unconsolidated", rock_classes("Quaternary alluvium") == ["unconsolidated"])
check("'Late Pliocene conglomerate and sandstone' -> sedimentary only", rock_classes("Late Pliocene conglomerate and sandstone") == ["sedimentary"])
check("'Glaciers' -> ice", rock_classes("Glaciers") == ["ice"])
check("'Batolito de Santa Bárbara' (Spanish batholith) -> plutonic", rock_classes("Batolito de Santa Bárbara") == ["plutonic"])
check("'Volcano-sedimentary' -> volcanic + sedimentary", rock_classes("Paleogene - Neogene Volcano-sedimentary") == ["volcanic", "sedimentary"])
check("unit name wins over a description that mentions other rocks",
      classify_unit({"name": "Late Pliocene conglomerate and sandstone", "descrip": "clasts of basalt and granite"}) == (["sedimentary"], "name+lith"))
check("description is only a labelled fallback",
      classify_unit({"name": "Jordan", "descrip": "basalt flows"}) == (["volcanic"], "description"))
tg = {"primary": ["volcanic"], "secondary": ["sedimentary"]}
check("geology verdicts holds / partial / breaks / unknown",
      [geology_verdict(["volcanic"], tg), geology_verdict(["sedimentary"], tg), geology_verdict(["plutonic"], tg), geology_verdict([], tg)]
      == ["holds", "partial", "breaks", "unknown"])
f, direction = feature_ratio(16.7, 28.4, "slope_deg")
check("ratio rule: slope 16.7 vs 28.4 deg is about x1.70 lower -> partial", abs(f - 1.70) < 0.01 and direction == "lower" and verdict_from_factor(f) == "partial")
check("ratio rule: x1.4 holds, x1.8 partial, x2.5 breaks",
      [verdict_from_factor(1.4), verdict_from_factor(1.8), verdict_from_factor(2.5)] == ["holds", "partial", "breaks"])
check("eps floor: roughness 0.5 m vs 2 m (both below the ~2 m noise floor) is NOT a break",
      verdict_from_factor(feature_ratio(0.5, 2.0, "roughness_m")[0]) != "breaks")
check("bare ground: 96% holds, 50% partial, 3% breaks", [bare_verdict(0.96), bare_verdict(0.5), bare_verdict(0.03)] == ["holds", "partial", "breaks"])
check("every target has a literature rock type with a citation",
      all(TARGET_GEOLOGY.get(t["id"], {}).get("citation") for t in TARGETS))

# 6. landform + land mask
yy2, xx2 = np.meshgrid(np.linspace(-1, 1, n), np.linspace(-1, 1, n))
dome = 1000 * np.exp(-3 * (xx2 ** 2 + yy2 ** 2))
check("a volcano shape is peak-like", landform_class(elevation_skew(dome)) == "peak-like")
check("a crater shape is basin-like", landform_class(elevation_skew(-dome)) == "basin-like")
check("a tilted plane is mixed/flat", landform_class(elevation_skew(0.1 * Xp)) == "mixed/flat")
check("landform verdicts holds / partial / breaks",
      [landform_verdict(-1, -1), landform_verdict(-1, 0), landform_verdict(-1, 1)] == ["holds", "partial", "breaks"])
check("land mask: Danakil Depression (below sea level) is land", bool(is_land(14.24, 40.30)))
check("land mask: Mount Cameroon is land", bool(is_land(4.217, 9.167)))
check("land mask: the central Pacific is not land", not bool(is_land(0.0, -150.0)))
print("\nAll self-tests passed ✓")

### 6 · Data readers
*Why:* each reader returns elevations (m) at a list of lat/lon points, reading only the part of the file it needs. `GeoGrid` keeps one latitude band of the global ETOPO map in memory for the fast scan. Its longitude wraps around, so windows crossing the date line work.

In [ ]:
import rasterio, requests
from rasterio.merge import merge
from rasterio.windows import Window

os.environ.update({
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES", "GDAL_HTTP_MULTIRANGE": "YES",
    "GDAL_HTTP_MAX_RETRY": "5", "GDAL_HTTP_RETRY_DELAY": "2", "VSI_CACHE": "TRUE",
})

_OPEN = {}
def open_raster(path):
    if path not in _OPEN:
        _OPEN[path] = rasterio.open(path)
    return _OPEN[path]

def to_float(raw, nodata, scale=1.0, offset=0.0):
    a = raw.astype("float64")
    if nodata is not None and not np.isnan(nodata):
        a[a == nodata] = np.nan
    return a * (scale if scale is not None else 1.0) + (offset if offset is not None else 0.0)

def copernicus_tile(lat_floor, lon_floor):
    name = (f"Copernicus_DSM_COG_30_{'N' if lat_floor >= 0 else 'S'}{abs(lat_floor):02d}_00_"
            f"{'E' if lon_floor >= 0 else 'W'}{abs(lon_floor):03d}_00_DEM")
    return f"/vsicurl/https://copernicus-dem-90m.s3.amazonaws.com/{name}/{name}.tif"

def read_earth_tiles(source, lats, lons, log=print):
    west, east = lons.min() - 0.02, lons.max() + 0.02
    south, north = lats.min() - 0.02, lats.max() + 0.02
    srcs = []
    for a in range(math.floor(south), math.floor(north) + 1):
        for b in range(math.floor(west), math.floor(east) + 1):
            try:
                srcs.append(rasterio.open(copernicus_tile(a, b)))
            except Exception:
                log(f"   Copernicus tile {a},{b} not available (ocean or withheld)")
    if not srcs:
        raise RuntimeError("no Copernicus tiles cover this location")
    mosaic, tr = merge(srcs, bounds=(west, south, east, north), nodata=-9999)
    for s in srcs:
        s.close()
    data = to_float(mosaic[0], -9999)
    return bilinear(data, (lats - tr.f) / tr.e - 0.5, (lons - tr.c) / tr.a - 0.5)

def read_global_file(source, lats, lons, log=print):
    src = open_raster(source["path"])
    if src.crs.is_geographic:
        xs, ys = (lons + 180) % 360 - 180, lats
    else:
        p = src.crs.to_dict()
        if p.get("proj") != "eqc":
            raise ValueError(f"unsupported projection {p}")
        radius = p.get("R") or p.get("a") or BODIES[source["body"]]["radius_m"]
        dlon = (lons - p.get("lon_0", 0.0) + 180) % 360 - 180
        xs = radius * math.cos(math.radians(p.get("lat_ts", 0.0))) * np.radians(dlon) + p.get("x_0", 0.0)
        ys = radius * np.radians(lats - p.get("lat_0", 0.0)) + p.get("y_0", 0.0)
    tr, H, W = src.transform, src.height, src.width
    cols = (xs - tr.c) / tr.a - 0.5
    rows = (ys - tr.f) / tr.e - 0.5
    r0 = max(int(np.floor(rows.min())) - 1, 0)
    r1 = min(int(np.ceil(rows.max())) + 2, H)
    if cols.max() - cols.min() > W / 2:
        c0, c1 = 0, W
    else:
        c0 = max(int(np.floor(cols.min())) - 1, 0)
        c1 = min(int(np.ceil(cols.max())) + 2, W)
    raw = src.read(1, window=Window(c0, r0, c1 - c0, r1 - r0))
    data = to_float(raw, src.nodata, src.scales[0], src.offsets[0])
    return bilinear(data, rows - r0, cols - c0)

READERS = {"earth_tiles": read_earth_tiles, "global_file": read_global_file}

def read_points(source_key, lats, lons, log=print):
    source = SOURCES[source_key]
    flat = READERS[source["kind"]](source, np.ravel(lats), np.ravel(lons), log)
    return np.asarray(flat).reshape(np.shape(lats))

def measure_window(lat, lon, body, source_key, pixel_m, k=None, log=print):
    """Same measurement as the main pipeline, at any pixel size. Returns (z, features or None, coverage)."""
    k = k or supersample_factor(SOURCES[source_key]["native_res_m"], pixel_m)
    X, Y = window_offsets(pixel_m, k)
    la, lo = aeqd_inverse(X, Y, lat, lon, BODIES[body]["radius_m"])
    z = block_average(read_points(source_key, la, lo, log), k)
    feats, cov = terrain_features(z, pixel_m)
    return z, feats, cov

class GeoGrid:
    """One latitude band of a global lat/lon raster in memory; bilinear sampling with longitude wrap."""
    def __init__(self, data, transform, row0, pad):
        self.data, self.tr, self.row0, self.pad = data, transform, row0, pad
    def sample(self, lats, lons):
        rows = (np.asarray(lats) - self.tr.f) / self.tr.e - 0.5 - self.row0
        cols = (np.asarray(lons) - self.tr.c) / self.tr.a - 0.5 + self.pad
        return bilinear(self.data, rows, cols)

def load_band(path, lat_north, lat_south, pad=256):
    src = open_raster(path)
    tr = src.transform
    r0 = max(int(math.floor((lat_north - tr.f) / tr.e)) - 2, 0)
    r1 = min(int(math.ceil((lat_south - tr.f) / tr.e)) + 2, src.height)
    raw = src.read(1, window=Window(0, r0, src.width, r1 - r0))
    data = to_float(raw, src.nodata).astype("float32")
    data = np.concatenate([data[:, -pad:], data, data[:, :pad]], axis=1)
    return GeoGrid(data, tr, r0, pad)

def nasa_power(lat, lon):
    r = requests.get("https://power.larc.nasa.gov/api/temporal/climatology/point",
                     params={"parameters": "T2M,T2M_MIN,T2M_MAX,PRECTOTCORR", "community": "AG",
                             "latitude": round(float(lat), 4), "longitude": round(float(lon), 4), "format": "JSON"},
                     timeout=90)
    r.raise_for_status()
    p = r.json()["properties"]["parameter"]
    val = lambda k: None if p[k]["ANN"] in (None, -999, -999.0) else float(p[k]["ANN"])
    return {"temp_mean_c": val("T2M"), "temp_min_c": val("T2M_MIN"), "temp_max_c": val("T2M_MAX"), "precip_mm_day": val("PRECTOTCORR")}

def macrostrat_geology(lat, lon):
    """Rock unit at a point from Macrostrat; tries detailed maps first, then coarser ones."""
    for scale in ("large", "medium", "small", "tiny"):
        r = requests.get("https://macrostrat.org/api/v2/geologic_units/map",
                         params={"lat": round(float(lat), 5), "lng": round(float(lon), 5), "scale": scale}, timeout=60)
        r.raise_for_status()
        js = r.json().get("success", {})
        units = js.get("data") or []
        if units:
            u = units[0]
            classes, class_source = classify_unit(u)
            return {"unit": u.get("name"), "lith": u.get("lith") or None, "age": u.get("best_int_name"),
                    "map_scale": scale, "classes": classes, "class_source": class_source,
                    "confidence": "low" if class_source == "description" else ("normal" if class_source else None),
                    "reference": (js.get("refs") or {}).get(str(u.get("source_id")))}
    return None

from rasterio.enums import Resampling
WORLDCOVER_CLASSES = {10: "tree", 20: "shrub", 30: "grass", 40: "cropland", 50: "built", 60: "bare_sparse",
                      70: "snow_ice", 80: "water", 90: "wetland", 95: "mangrove", 100: "moss_lichen"}

def worldcover_tile(lat3, lon3):
    name = (f"ESA_WorldCover_10m_2021_v200_{'N' if lat3 >= 0 else 'S'}{abs(lat3):02d}"
            f"{'E' if lon3 >= 0 else 'W'}{abs(lon3):03d}_Map")
    return f"/vsicurl/https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/{name}.tif"

def land_cover_fractions(lat, lon, res_deg=0.0009):
    """Share of each ESA WorldCover class in the window's lat/lon box, read at ~100 m from the 10 m map."""
    half_lat = (WINDOW_KM / 2) / EARTH_KM_PER_DEG
    half_lon = half_lat / max(math.cos(math.radians(lat)), 0.01)
    west, east, south, north = lon - half_lon, lon + half_lon, lat - half_lat, lat + half_lat
    srcs = []
    for a in range(math.floor(south / 3) * 3, math.floor(north / 3) * 3 + 1, 3):
        for b in range(math.floor(west / 3) * 3, math.floor(east / 3) * 3 + 1, 3):
            try:
                srcs.append(rasterio.open(worldcover_tile(a, b)))
            except Exception:
                pass                                   # no tile = open ocean
    if not srcs:
        return None
    mosaic, _ = merge(srcs, bounds=(west, south, east, north), res=(res_deg, res_deg), nodata=0,
                      resampling=Resampling.nearest)
    for s in srcs:
        s.close()
    v = mosaic[0]
    n_valid = int((v > 0).sum())
    if n_valid == 0:
        return None
    return {name: round(float((v == code).sum()) / n_valid, 4) for code, name in WORLDCOVER_CLASSES.items()}

### 7 · Data access + connection test
*Why:* the quick scan streams ETOPO. The full scan downloads it once into Colab (and checks the file size, so a broken download can't silently corrupt the scan). Then four known facts are tested: a mountain is high, the ocean is negative, the South Pole sits on thick ice, and the Sahara has none.

In [ ]:
def etopo_path(kind):
    url = ETOPO_URLS[kind]
    if QUICK_SCAN:
        return "/vsicurl/" + url
    local = f"/content/etopo_2022_30s_{kind}.tif"
    expected = int(requests.head(url, allow_redirects=True, timeout=60).headers["Content-Length"])
    if not (os.path.exists(local) and os.path.getsize(local) == expected):
        t0 = time.time()
        print(f"Downloading ETOPO 2022 {kind} ({expected / 1e9:.2f} GB) into Colab ...")
        subprocess.run(["curl", "-sS", "-L", "--retry", "8", "--retry-delay", "3", "-C", "-", "-o", local, url], check=True)
        print(f"   done in {time.time() - t0:.0f}s")
    size = os.path.getsize(local)
    check(f"ETOPO {kind} file complete ({size:,} bytes)", size == expected)
    return local

SOURCES["etopo-2022-30s"]["path"] = etopo_path("surface")
SOURCES["etopo-2022-30s-bed"]["path"] = etopo_path("bed")

def etopo_point(key, lat, lon):
    return float(read_points(key, np.array([lat]), np.array([lon]))[0])

cam = etopo_point("etopo-2022-30s", 4.217, 9.167)
check(f"Mount Cameroon area is high: {cam:,.0f} m (expected 2,500-4,200 m at 1 km)", 2500 < cam < 4200)
pac = etopo_point("etopo-2022-30s", 0.0, -150.0)
check(f"Central Pacific is ocean: {pac:,.0f} m (expected negative)", pac < 0)
sp_ice = etopo_point("etopo-2022-30s", -89.99, 0.0) - etopo_point("etopo-2022-30s-bed", -89.99, 0.0)
check(f"South Pole ice thickness: {sp_ice:,.0f} m (expected > 1,000 m)", sp_ice > 1000)
sah_ice = etopo_point("etopo-2022-30s", 23.0, 12.0) - etopo_point("etopo-2022-30s-bed", 23.0, 12.0)
check(f"Sahara ice thickness: {sah_ice:,.1f} m (expected ~0)", abs(sah_ice) < 1)

# Geology service: a failure here is a warning only (the scan does not depend on it)
for label, la, lo, expect in [("Mauna Kea", 19.821, -155.468, "volcanic"), ("Yungay, Atacama", -24.08, -70.0, "plutonic")]:
    try:
        g = macrostrat_geology(la, lo)
        ok = bool(g) and expect in g["classes"]
        print(("PASS  " if ok else "WARN  ") + f"geology at {label}: {g['unit'] if g else None} -> {g['classes'] if g else None} (expected {expect})")
    except Exception as e:
        print(f"WARN  geology service unavailable ({type(e).__name__}); candidates will show geology 'unknown'")

# Land cover: a glaciated window from the previous global run, and the Sahara as a bare-ground control
for label, la, lo, key, test in [("St. Elias glaciers, Alaska/Yukon", 58.90, -137.70, "snow_ice", lambda v: v > MAX_GLACIER_FRACTION),
                                 ("Sahara", 23.0, 12.0, "bare_sparse", lambda v: v > 0.8)]:
    try:
        lc = land_cover_fractions(la, lo)
        v = (lc or {}).get(key)
        print(("PASS  " if v is not None and test(v) else "WARN  ") + f"land cover at {label}: {key} = {v}")
    except Exception as e:
        print(f"WARN  land cover unavailable ({type(e).__name__}: {e}); glacier filter will be skipped")

### 8 · Measure the Moon/Mars targets at both resolutions
*Why:* every target is measured exactly like the Earth windows: at 1 km for the screen, at 500 m for the confirmation.

In [ ]:
import pandas as pd
for t in TARGETS:
    src_key = TERRAIN_SOURCE[t["body"]]
    t0 = time.time()
    _, t["feat_a"], cov_a = measure_window(t["lat"], t["lon"], t["body"], src_key, PIXEL_A_M)
    zb, t["feat_b"], cov_b = measure_window(t["lat"], t["lon"], t["body"], src_key, PIXEL_B_M)
    t["skew_b"] = elevation_skew(zb)
    t["landform_b"] = landform_class(t["skew_b"])
    check(f"{t['id']}: both resolutions measured (coverage {cov_a:.2f} / {cov_b:.2f}) [{time.time() - t0:.1f}s]",
          t["feat_a"] is not None and t["feat_b"] is not None)
pd.DataFrame([{"target": t["id"], **{f"{k} @1km": round(t["feat_a"][k], 2) for k in FEATURE_KEYS},
               **{f"{k} @500m": round(t["feat_b"][k], 2) for k in FEATURE_KEYS},
               "skew": round(t["skew_b"], 2), "landform": t["landform_b"]} for t in TARGETS])

### 9 · Stage A: scan Earth's land
*Why:* every non-overlapping 20 km window on land is measured. Land is decided by an **offline land mask**, not by "elevation > 0": land below sea level (Danakil Depression, Dead Sea shore, Death Valley, Turpan) is real land — and often prime analog terrain — while a window with more than 5% ocean/lake cells is rejected. Ice-sheet cover (surface minus bedrock) is recorded for every window.

In [ ]:
t_scan = time.time()
lat_lo, lat_hi = QUICK_LAT_RANGE if QUICK_SCAN else LAT_RANGE
dlat = SPACING_KM / EARTH_KM_PER_DEG
lat_centres = np.arange(lat_lo + dlat / 2, lat_hi, dlat)
Xa, Ya = window_offsets(PIXEL_A_M, 1)
na = Xa.shape[0]
probe = [(i, j) for i in (0, na // 2, na - 1) for j in (0, na // 2, na - 1)]
margin = (WINDOW_KM * 0.75) / EARTH_KM_PER_DEG + 0.05
cols_out = {k: [] for k in ["lat", "lon", *FEATURE_KEYS, "coverage", "water", "ice"]}
n_probed = 0

for north in np.arange(lat_hi, lat_lo, -BAND_DEG):
    south = max(north - BAND_DEG, lat_lo)
    rows_here = lat_centres[(lat_centres < north) & (lat_centres >= south)]
    if rows_here.size == 0:
        continue
    t0 = time.time()
    surf = load_band(SOURCES["etopo-2022-30s"]["path"], min(north + margin, 90), max(south - margin, -90))
    bed = load_band(SOURCES["etopo-2022-30s-bed"]["path"], min(north + margin, 90), max(south - margin, -90))
    kept_band = 0
    for lat0 in rows_here:
        plat, plon = aeqd_inverse(Xa, Ya, lat0, 0.0, BODIES["earth"]["radius_m"])
        nlon = max(1, int(round(360 * math.cos(math.radians(lat0)) * EARTH_KM_PER_DEG / SPACING_KM)))
        lon_c = -180 + (np.arange(nlon) + 0.5) * 360 / nlon
        n_probed += nlon
        pr = np.array([plat[i, j] for i, j in probe]); pl = np.array([plon[i, j] for i, j in probe])
        probe_lon = (lon_c[:, None] + pl[None, :] + 180) % 360 - 180
        probe_lat = np.broadcast_to(pr[None, :], probe_lon.shape)
        pland = is_land(probe_lat.ravel(), probe_lon.ravel()).reshape(probe_lon.shape)
        keep = pland.any(axis=1)                      # at least one probe point on land (mask, not elevation)
        if not keep.any():
            continue
        lc = lon_c[keep]
        Lon = (lc[:, None, None] + plon[None] + 180) % 360 - 180
        Lat = np.broadcast_to(plat[None], Lon.shape)
        land = is_land(Lat.ravel(), Lon.ravel()).reshape(Lon.shape)
        water = 1.0 - land.reshape(len(lc), -1).mean(axis=1)     # share of non-land (ocean/lake) cells
        zs = surf.sample(Lat.ravel(), Lon.ravel()).reshape(Lon.shape).astype("float64")
        zb = bed.sample(Lat.ravel(), Lon.ravel()).reshape(Lon.shape).astype("float64")
        ice = ((zs - zb) > ICE_MIN_THICKNESS_M).reshape(len(lc), -1).mean(axis=1)
        ok = water <= MAX_WATER_FRACTION
        if not ok.any():
            continue
        f, cov = terrain_features_batch(zs[ok], PIXEL_A_M)
        cols_out["lat"].append(np.full(ok.sum(), lat0)); cols_out["lon"].append(lc[ok])
        for k in FEATURE_KEYS:
            cols_out[k].append(f[k])
        cols_out["coverage"].append(cov); cols_out["water"].append(water[ok]); cols_out["ice"].append(ice[ok])
        kept_band += int(ok.sum())
    print(f"band {south:6.1f}..{north:6.1f}: {kept_band:6d} land windows  [{time.time() - t0:.0f}s]")
    del surf, bed

E = {k: np.concatenate(v) if v else np.array([]) for k, v in cols_out.items()}
valid = E["coverage"] >= MIN_COVERAGE
if EXCLUDE_ICE_SHEETS:
    valid &= E["ice"] <= MAX_ICE_FRACTION
E = {k: v[valid] for k, v in E.items()}
N = E["lat"].size
print(f"\nScanned {n_probed:,} windows, kept {N:,} valid land windows in {time.time() - t_scan:.0f}s "
      f"({'QUICK band ' + str(QUICK_LAT_RANGE) if QUICK_SCAN else 'whole Earth'})")
check("scan produced land windows", N > 100)
check("no missing feature values among valid windows", all(np.isfinite(E[k]).all() for k in FEATURE_KEYS))

LE = log_features(E)
SCALE = robust_scale(LE)
W_VEC = weight_vector()
print("Earth variability (robust std of log10 features):", dict(zip(FEATURE_KEYS, np.round(SCALE, 3))))

### 10 · Stage A ranking + surface pre-filter + stability test
*Why:*
- **Ranking:** each target is compared with every land window, and the score is the percentile.
- **Surface pre-filter:** walking the ranking best-first, a candidate is kept only if ESA WorldCover shows it is at most 30% snow/ice, at most 20% water, and at least 30% bare ground (`MIN_BARE_FRACTION`; set 0 to disable). The Moon and Mars are 100% bare rock and regolith, so a forest with the right relief is not a usable analog. Before, this was only checked *after* selection, so vegetated candidates crowded bare ones out of the list.
- **Non-maximum suppression:** candidates stay at least 150 km apart, so one mountain range can't fill the whole list.
- **Stability:** feature weights are a human choice, so we redraw them randomly 200 times. For each candidate we record its rank on Earth every time: the **median rank**, the **worst rank**, and the share of draws in which it stays in Earth's **top 0.1%**. A small worst rank means the candidate doesn't depend on our weight choice.
- This screen is a **filter**. Candidates are nearly tied at 1 km, so the **final order comes from the 500 m confirmation** (cell 11).

In [ ]:
rng = np.random.default_rng(SEED)
WDRAWS = rng.dirichlet(np.full(len(FEATURE_KEYS), STABILITY_ALPHA), size=STABILITY_DRAWS)

LC_CACHE = {}
def window_land_cover(i):
    """ESA WorldCover shares for window i, fetched once and reused by Stage B."""
    if i not in LC_CACHE:
        try:
            LC_CACHE[i] = land_cover_fractions(float(E["lat"][i]), float(E["lon"][i]))
        except Exception:
            LC_CACHE[i] = None
    return LC_CACHE[i]

def surface_ok(i):
    """Pre-filter: not a glacier, not a lake, bare enough for a Moon/Mars surface analog.
    None (no data / service down) passes here; Stage B decides with its own safety net."""
    lc = window_land_cover(i)
    if lc is None:
        return True
    snow = lc.get("snow_ice", 0.0)
    water = lc.get("water", 0.0) + lc.get("wetland", 0.0)
    bare = lc.get("bare_sparse", 0.0) + lc.get("moss_lichen", 0.0)
    return snow <= MAX_GLACIER_FRACTION and water <= MAX_WATER_COVER and bare >= MIN_BARE_FRACTION

for t in TARGETS:
    lt = log_features(t["feat_a"])
    d = weighted_distance(LE, lt, SCALE, W_VEC)
    t["d_sorted"] = np.sort(d)
    order = np.argsort(d)
    chosen, looked_up, rejected = [], 0, 0
    for i in order:                                   # best-first walk: separation + surface checks
        if len(chosen) == N_CANDIDATES:
            break
        i = int(i)
        if any(haversine_km(E["lat"][i], E["lon"][i], E["lat"][j], E["lon"][j]) < MIN_SEPARATION_KM for j in chosen):
            continue
        if MIN_BARE_FRACTION > 0:
            if looked_up >= MAX_PREFILTER_TRIES:
                break
            looked_up += 1
            if not surface_ok(i):
                rejected += 1
                continue
        chosen.append(i)
    if len(chosen) < N_CANDIDATES:
        print(f"   note: only {len(chosen)} candidates pass the surface pre-filter within {looked_up} land-cover lookups")
    Z2 = ((LE - lt) / SCALE) ** 2
    top_n = max(1, int(TOP_FRACTION * N))
    ranks = []
    for s0 in range(0, STABILITY_DRAWS, 25):
        D = np.sqrt(Z2 @ WDRAWS[s0:s0 + 25].T)                       # (N windows, draws)
        Dc = D[chosen]                                               # (candidates, draws)
        ranks.append(np.stack([(D < Dc[n_][None, :]).sum(axis=0) + 1 for n_ in range(len(chosen))]))
    ranks = np.concatenate(ranks, axis=1)                            # (candidates, all draws)
    t["cands"] = [{"idx": i, "lat": float(E["lat"][i]), "lon": float(E["lon"][i]), "d_a": float(d[i]),
                   "pct_a": float(percentile_closer(t["d_sorted"], d[i])),
                   "stability": float(np.mean(ranks[n_] <= top_n)),
                   "rank_median": int(np.median(ranks[n_])), "rank_worst": int(ranks[n_].max()),
                   "screen_rank": int(np.searchsorted(t["d_sorted"], d[i], side="left") + 1)}
                  for n_, i in enumerate(chosen)]
    print(f"\n{t['name']} ({t['body']}): best {len(chosen)} of {N:,} windows "
          f"({rejected} rejected by the surface pre-filter in {looked_up} lookups)")
    print(pd.DataFrame([{k: (round(v, 3) if isinstance(v, float) else v) for k, v in c.items() if k != "idx"}
                        for c in t["cands"]]).to_string(index=False))

### 11 · Stage B: confirm at 500 m, add climate, name the place
*Why:* the screen used 1 km ETOPO data, so each candidate is **re-measured independently** with 500 m Copernicus data (the main pipeline's method) and compared with the target's 500 m features. Each feature gets a verdict from a plain ratio:
- **holds:** Earth and target values within **×1.5** of each other
- **partial:** within **×2**
- **breaks:** more than ×2 apart

It's a plain ratio that anyone can check by dividing the two numbers (after adding the eps noise floor, so differences the data can't resolve don't count). Bare ground (ESA WorldCover) gets its own verdict, because Moon and Mars surfaces are 100% bare: ≥70% holds, ≥30% partial, otherwise breaks.

Then four context layers are added:
- **Landform:** the skewness of the window's 500 m elevations says whether the terrain is **basin-like** (crater, depression), **peak-like** (volcano, massif) or mixed/flat, and is compared with the target's landform. Relief, slope and roughness can't tell a crater from a volcano of the same size; this can.
- **Geology:** the rock unit at the candidate (Macrostrat, which compiles published geological maps), sorted into volcanic / plutonic / metamorphic / sedimentary / unconsolidated. It is compared with the target's rock type **from published papers**: *holds* = same main rock class, *partial* = a secondary class, *breaks* = neither. Rock classes come from the map unit's **name and rock-type field**; the long description is only a fallback, marked `?` (low confidence).
- **Novelty:** distance to the nearest of our 14 established analog sites (under 50 km = already established).
- **Climate:** NASA POWER, for context only. Moon/Mars temperatures come from published measurements via the pipeline's `LITERATURE` table (see `targets.json`); they are shown, not scored.
- **Exclusions (safety net):** windows over 30% snow/ice (mountain glaciers) or 20% water (lakes the land mask misses) are marked `excluded` and not ranked — most never get this far thanks to the pre-filter in cell 10.

In [ ]:
import reverse_geocoder as rg
try:
    import pycountry
except Exception:
    pycountry = None

def country_name(cc):
    try:
        return pycountry.countries.get(alpha_2=cc).name if pycountry else cc
    except Exception:
        return cc

for t in TARGETS:
    print(f"\n▶ confirming {t['name']}")
    lt_b = log_features(t["feat_b"])
    places = rg.search([(c["lat"], c["lon"]) for c in t["cands"]], mode=1)
    for c, pl in zip(t["cands"], places):
        c["place"] = {"name": pl["name"], "admin1": pl["admin1"], "country_code": pl["cc"], "country": country_name(pl["cc"]),
                      "distance_km": round(float(haversine_km(c["lat"], c["lon"], float(pl["lat"]), float(pl["lon"]))), 1)}
        dk = [float(haversine_km(c["lat"], c["lon"], s[2], s[3])) for s in KNOWN_SITES]
        nk = int(np.argmin(dk))
        c["nearest_known_site"] = {"id": KNOWN_SITES[nk][0], "name": KNOWN_SITES[nk][1], "distance_km": round(dk[nk], 1)}
        c["near_known_site"] = KNOWN_SITES[nk][0] if dk[nk] < 50 else None
        c["features_1km"] = {k: float(E[k][c["idx"]]) for k in FEATURE_KEYS}
        c["water_fraction"], c["ice_fraction"] = float(E["water"][c["idx"]]), float(E["ice"][c["idx"]])
        c["errors"] = []
        try:
            z, fb, cov = measure_window(c["lat"], c["lon"], "earth", "copernicus-glo90", PIXEL_B_M, log=lambda m: None)
            if fb is None:
                raise RuntimeError(f"coverage {cov:.2f}")
            c["features_500m"] = fb
            c["below_sea_fraction_500m"] = float(np.mean(z[np.isfinite(z)] <= 0))   # info only: can be LAND below sea level
            skew = elevation_skew(z)
            c["landform"] = {"earth_skew": round(skew, 3), "target_skew": round(t["skew_b"], 3),
                             "earth": landform_class(skew), "target": t["landform_b"],
                             "verdict": landform_verdict(skew, t["skew_b"])}
            lb = log_features(fb)
            c["d_b"] = float(weighted_distance(lb, lt_b, SCALE, W_VEC))
            zf = (lb - lt_b) / SCALE
            c["comparison"] = {}
            for j, k in enumerate(FEATURE_KEYS):
                factor, direction = feature_ratio(fb[k], t["feat_b"][k], k)
                c["comparison"][k] = {"earth": round(fb[k], 2), "target": round(t["feat_b"][k], 2),
                                      "factor": round(float(factor), 3), "direction": direction,
                                      "z": round(float(zf[j]), 2), "verdict": verdict_from_factor(factor)}
        except Exception as e:
            c["features_500m"], c["d_b"], c["comparison"], c["landform"] = None, None, None, None
            c["errors"].append(f"stage B: {type(e).__name__}: {e}")
        try:
            c["climate"] = nasa_power(c["lat"], c["lon"])
        except Exception as e:
            c["climate"] = None
            c["errors"].append(f"climate: {type(e).__name__}: {e}")
        try:
            geo = macrostrat_geology(c["lat"], c["lon"])
            if geo:
                geo["verdict"] = geology_verdict(geo["classes"], TARGET_GEOLOGY.get(t["id"]))
            c["geology"] = geo
        except Exception as e:
            c["geology"] = None
            c["errors"].append(f"geology: {type(e).__name__}: {e}")
        c["excluded"] = None
        try:
            c["land_cover"] = LC_CACHE.get(c["idx"]) or land_cover_fractions(c["lat"], c["lon"])
            lcov = c["land_cover"] or {}
            snow = lcov.get("snow_ice", 0.0)
            wat = lcov.get("water", 0.0) + lcov.get("wetland", 0.0)
            if c["land_cover"]:
                bare = lcov.get("bare_sparse", 0.0) + lcov.get("moss_lichen", 0.0)
                c["bare_ground"] = {"fraction": round(bare, 4), "verdict": bare_verdict(bare)}
            else:
                c["bare_ground"] = None
            if snow > MAX_GLACIER_FRACTION:
                c["excluded"] = f"glaciated: {snow:.0%} snow/ice in ESA WorldCover"
            elif wat > MAX_WATER_COVER:
                c["excluded"] = f"water body: {wat:.0%} water in ESA WorldCover"
        except Exception as e:
            c["land_cover"], c["bare_ground"] = None, None
            c["errors"].append(f"land cover: {type(e).__name__}: {e}")
        time.sleep(0.2)
    ok_c = [c for c in t["cands"] if c["d_b"] is not None and not c["excluded"]]
    for r_, c in enumerate(sorted(ok_c, key=lambda c: c["d_b"]), start=1):
        c["confirm_rank"] = r_
    for c in t["cands"]:
        c.setdefault("confirm_rank", None)
    t["cands"].sort(key=lambda c: (c["confirm_rank"] is None, c["confirm_rank"] or 0))   # final order = 500 m result
    print(pd.DataFrame([{"screen": c["screen_rank"], "confirm": c["confirm_rank"], "lat": round(c["lat"], 2), "lon": round(c["lon"], 2),
                         "near": f"{c['place']['name']}, {c['place']['country_code']} ({c['place']['distance_km']} km)",
                         "pct_A": round(c["pct_a"], 3), "stab": round(c["stability"], 2), "worst": c["rank_worst"],
                         "terrain": " / ".join(v["verdict"] for v in c["comparison"].values()) if c["comparison"] else "-",
                         "landform": (c["landform"] or {}).get("verdict", "-"),
                         "rock": (c["geology"] or {}).get("unit") or "-",
                         "geology": (c["geology"] or {}).get("verdict", "unknown") + ("?" if (c["geology"] or {}).get("confidence") == "low" else ""),
                         "bare%": round(100 * (c["bare_ground"] or {}).get("fraction", 0)),
                         "surface": (c["bare_ground"] or {}).get("verdict", "-"),
                         "snow%": round(100 * (c["land_cover"] or {}).get("snow_ice", 0)),
                         "known_km": c["nearest_known_site"]["distance_km"],
                         "excluded": c["excluded"] or "",
                         "errors": "; ".join(c["errors"])} for c in t["cands"]]).to_string(index=False))
    n_ex = sum(1 for c in t["cands"] if c["excluded"])
    print(f"   {n_ex} candidate(s) excluded (glacier/water); '?' after a geology verdict = classified from the long description (low confidence)")
    tgeo = TARGET_GEOLOGY.get(t["id"], {})
    print(f"   target rock type: {tgeo.get('primary')} (secondary {tgeo.get('secondary')}) per {tgeo.get('citation', 'n/a')[:80]}")
    print(f"   target landform: {t['landform_b']} (skew {t['skew_b']:.2f})")

### 12 · Validation: how much can we trust this?
1. **Dataset agreement:** the same random windows measured with ETOPO and with Copernicus (independent processing, same 1 km grid). High correlation = the screen isn't an artefact of one dataset. *Recall@1%:* how often a window measured with Copernicus finds **itself** within the top 1% of the ETOPO screen.
2. **Stage A vs Stage B:** do the 500 m measurements agree with the 1 km ranking? (Spearman rank correlation of the distances.)
3. **Established analog sites:** where each known site ranks for each target (percentile of Earth's land). This shows which famous analogs are *terrain* analogs and which are analogs for other reasons (chemistry, climate, biology).

In [ ]:
from scipy.stats import pearsonr, spearmanr

# 1. dataset agreement
sel = np.where((np.abs(E["lat"]) < 60) & (E["ice"] == 0))[0]
pick = rng.choice(sel, size=min(N_AGREEMENT, sel.size), replace=False)
pairs = []
for i in pick:
    try:
        _, fc, _ = measure_window(float(E["lat"][i]), float(E["lon"][i]), "earth", "copernicus-glo90", PIXEL_A_M, log=lambda m: None)
        if fc:
            pairs.append((i, fc))
    except Exception as e:
        print("   skipped one window:", type(e).__name__, e)
agreement = {"n_windows": len(pairs)}
if len(pairs) >= 10:
    LC = np.array([log_features(fc) for _, fc in pairs])
    LEp = LE[[i for i, _ in pairs]]
    for j, k in enumerate(FEATURE_KEYS):
        r = float(pearsonr(LEp[:, j], LC[:, j])[0])
        mad = float(np.median(np.abs(LEp[:, j] - LC[:, j])))
        agreement[k] = {"pearson_r_log": round(r, 3), "median_abs_log10_diff": round(mad, 3),
                        "rating": "strong" if r >= 0.9 else ("moderate" if r >= 0.7 else "weak")}
        print(f"{k:12s} r = {r:.3f} ({agreement[k]['rating']}), median |Δlog10| = {mad:.3f} (factor {10 ** mad:.2f})")
    hits = []
    for (i, fc), lc in zip(pairs, LC):
        d_self = weighted_distance(LE, lc, SCALE, W_VEC)
        hits.append(np.mean(d_self < d_self[i]) <= RECALL_FRACTION)
    agreement["recall_at_top1pct"] = round(float(np.mean(hits)), 3)
    print(f"recall@1%: {agreement['recall_at_top1pct']:.0%} of windows measured with Copernicus find themselves in the ETOPO top 1%")
else:
    print("Not enough windows for the agreement test")

# 2. Stage A vs Stage B
stage_agreement = {}
for t in TARGETS:
    ok_c = [c for c in t["cands"] if c["d_b"] is not None]
    if len(ok_c) >= 4:
        rho = float(spearmanr([c["d_a"] for c in ok_c], [c["d_b"] for c in ok_c])[0])
        stage_agreement[t["id"]] = round(rho, 3)
print("\nStage A vs B rank agreement (Spearman):", stage_agreement)

# 3. established analog sites
known_rows, known_pct = [], {}
for sid, sname, la, lo in KNOWN_SITES:
    try:
        _, fk, _ = measure_window(la, lo, "earth", "etopo-2022-30s", PIXEL_A_M, k=1, log=lambda m: None)
    except Exception as e:
        fk = None
        print("   ", sid, type(e).__name__, e)
    row = {"site": sname}
    known_pct[sid] = {}
    for t in TARGETS:
        if fk is None:
            row[t["id"]] = None; continue
        p = float(percentile_closer(t["d_sorted"], weighted_distance(log_features(fk), log_features(t["feat_a"]), SCALE, W_VEC)))
        row[t["id"]] = round(p, 1); known_pct[sid][t["id"]] = round(p, 2)
    known_rows.append(row)
print("\nEstablished sites: % of Earth's land windows they beat, per target (100 = best possible)")
pd.DataFrame(known_rows)

### 13 · Map
*Why:* a visual sanity check. Grey = every valid land window scanned; coloured = confirmed candidates for each target; black crosses = established analog sites.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 7))
ax.scatter(E["lon"], E["lat"], s=0.2, c="#bbbbbb", lw=0)
colors = plt.cm.tab10(np.arange(len(TARGETS)))
for col, t in zip(colors, TARGETS):
    ax.scatter([c["lon"] for c in t["cands"]], [c["lat"] for c in t["cands"]], s=40, color=col, edgecolor="k", lw=0.5, label=t["name"])
ax.scatter([s[3] for s in KNOWN_SITES], [s[2] for s in KNOWN_SITES], marker="x", c="k", s=30, label="established sites")
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90); ax.set_aspect("equal")
ax.legend(loc="lower left", fontsize=8); ax.set_title("GeoProxy discovery: confirmed candidates per target")
plt.show()

### 14 · Build and save `discoveries.json`
*Why:* the web app reads this file. Each candidate carries its screen score, stability, the 500 m confirmation, per-feature verdicts, climate, the nearest town and whether it's already an established site. Novelty is stated honestly: "not in our list of established sites" is **not** the same as "never studied". Check the literature before claiming a discovery.

In [ ]:
from datetime import datetime, timezone

def clean(c):
    out = {k: v for k, v in c.items() if k not in ("idx",)}
    out["lat"], out["lon"] = round(c["lat"], 4), round(c["lon"], 4)
    nk = c["nearest_known_site"]
    out["novelty"] = (f"established analog site ({nk['name']})" if c["near_known_site"] else
                      f"{nk['distance_km']:.0f} km from the nearest established site in GeoProxy's list ({nk['name']}); "
                      "not yet checked against the scientific literature")
    return out

discoveries = {
    "schema_version": "0.2",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "scope": {"mode": "quick" if QUICK_SCAN else "global", "lat_range": list(QUICK_LAT_RANGE if QUICK_SCAN else LAT_RANGE),
              "windows_scanned": int(n_probed), "valid_land_windows": int(N)},
    "method": {
        "stage_a": {"dataset": "etopo-2022-30s", "pixel_m": PIXEL_A_M, "window_km": WINDOW_KM, "spacing_km": SPACING_KM,
                    "land_rule": f"land is decided by the GLOBE land mask (globe-land-mask source), NOT by elevation, so land below sea level is kept; windows with more than {MAX_WATER_FRACTION:.0%} non-land cells are rejected",
                    "max_water_fraction": MAX_WATER_FRACTION, "ice_sheets_excluded": EXCLUDE_ICE_SHEETS,
                    "ice_rule": f"surface minus bedrock > {ICE_MIN_THICKNESS_M} m on more than {MAX_ICE_FRACTION:.0%} of the window"},
        "stage_b": {"dataset": "copernicus-glo90", "pixel_m": PIXEL_B_M},
        "features": FEATURE_KEYS, "transform": "log10(value + eps)", "eps": LOG_EPS,
        "eps_rationale": "eps values approximate measurement noise floors (DEM vertical accuracy, laser-track interpolation); value differences smaller than eps are not treated as meaningful",
        "earth_variability_scale": dict(zip(FEATURE_KEYS, [round(float(s), 4) for s in SCALE])),
        "weights": dict(zip(FEATURE_KEYS, [round(float(w), 4) for w in W_VEC])),
        "score": "percentile of Earth's valid land windows that are less similar (100 = most similar on Earth)",
        "stability": f"share of {STABILITY_DRAWS} random weight sets (Dirichlet alpha={STABILITY_ALPHA}) in which the window stays in Earth's top {TOP_FRACTION * 100:g}%; rank_median / rank_worst = its rank on Earth across those draws",
        "ranking": "candidates are pre-selected by the 1 km screen (after the surface pre-filter) and ordered by the 500 m confirmation distance (rank = confirm_rank)",
        "surface_prefilter": (f"walking the screen ranking best-first, a candidate is kept only if ESA WorldCover shows <= {MAX_GLACIER_FRACTION:.0%} snow/ice, "
                              f"<= {MAX_WATER_COVER:.0%} water and >= {MIN_BARE_FRACTION:.0%} bare ground (MIN_BARE_FRACTION = 0 disables)") if MIN_BARE_FRACTION > 0 else "disabled",
        "dataset_agreement_caveat": "ETOPO 2022 land heights are largely derived from Copernicus DEM, so the agreement test checks processing consistency across products and resolutions, not fully independent measurements",
        "verdicts": {"holds": f"Earth and target within x{FACTOR_HOLD} of each other", "partial": f"within x{FACTOR_PARTIAL}",
                     "breaks": f"more than x{FACTOR_PARTIAL} apart",
                     "factor": "max(r, 1/r) with r = (earth + eps) / (target + eps)",
                     "z": "difference of log features divided by Earth's natural variability (used for the overall distance, not the verdict)"},
        "verdict_rule": {"factor_hold": FACTOR_HOLD, "factor_partial": FACTOR_PARTIAL,
                         "bare_hold": BARE_HOLD, "bare_partial": BARE_PARTIAL},
        "bare_ground": f"share of bare/sparse vegetation plus moss/lichen in ESA WorldCover; Moon and Mars are 100% bare, so >= {BARE_HOLD:.0%} holds, >= {BARE_PARTIAL:.0%} partial, otherwise breaks",
        "landform": {"measure": "skewness of the window's 500 m elevations",
                     "classes": f"basin-like (skew <= {SKEW_BASIN}: crater, caldera, depression), peak-like (skew >= {SKEW_PEAK}: volcano, massif), otherwise mixed/flat",
                     "verdicts": {"holds": "same landform class as the target", "partial": "one of the two is mixed/flat", "breaks": "basin vs peak"},
                     "why": "relief, slope and roughness are magnitude-only and cannot tell a crater from a volcano of the same size"},
        "min_separation_km": MIN_SEPARATION_KM,
        "geology": {"earth": "Macrostrat map unit at the window centre, classified by keyword into rock classes",
                    "targets": "rock classes from published papers (see each target's citation)",
                    "classes": list(ROCK_CLASSES), "verdicts": {"holds": "shares the target's primary rock class",
                    "partial": "shares a secondary class", "breaks": "shares neither", "unknown": "no map data"}},
        "novelty": "distance to the nearest of GeoProxy's 14 established analog sites; < 50 km = established",
        "land_cover": f"ESA WorldCover class shares over the window; candidates with > {MAX_GLACIER_FRACTION:.0%} snow/ice or > {MAX_WATER_COVER:.0%} water are excluded (excluded field) and not ranked",
        "geology_confidence": "low when the rock class was found only in the map unit's long description",
    },
    "validation": {"dataset_agreement_1km": agreement, "stage_a_vs_b_spearman": stage_agreement, "established_sites_percentile": known_pct},
    "sources": {k: {f: v for f, v in s.items() if f not in ("path", "kind")} for k, s in SOURCES.items()},
    "targets": [{"id": t["id"], "name": t["name"], "body": t["body"], "lat": t["lat"], "lon": t["lon"],
                 "features_1km": t["feat_a"], "features_500m": t["feat_b"],
                 "skew_500m": round(t["skew_b"], 3), "landform_500m": t["landform_b"],
                 "geology": TARGET_GEOLOGY.get(t["id"]),
                 "candidates": [clean(c) for c in t["cands"]]} for t in TARGETS],
}

out_dir = "/content/output"
if SAVE_TO_DRIVE and not QUICK_SCAN:          # the quick test never asks for Drive access
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = DRIVE_OUT
    except Exception as e:
        print("Drive not available, saving locally:", e)
os.makedirs(out_dir, exist_ok=True)
name = "discoveries.json" if not QUICK_SCAN else "discoveries_quick.json"
path = os.path.join(out_dir, name)
with open(path, "w", encoding="utf-8") as fh:
    json.dump(discoveries, fh, indent=2, ensure_ascii=False)
check(f"{name} written ({os.path.getsize(path):,} bytes) and re-readable", json.load(open(path, encoding="utf-8"))["schema_version"] == "0.2")
import hashlib
print("sha256:", hashlib.sha256(open(path, "rb").read()).hexdigest())   # lets anyone verify a copied file is identical
if AUTO_DOWNLOAD and not QUICK_SCAN:
    try:
        from google.colab import files
        files.download(path)
    except Exception as e:
        print("download skipped:", e)
print("\nRUN COMPLETE")